# 04 - EDA with SQL
**IBM Applied Data Science Capstone - SpaceX Falcon 9 First-Stage Landing Prediction**

Author: Pranav Vyankatesh Jagdale

**Goal:** load the launch table into SQLite and answer the lab questions with SQL.

> Run every cell in order. Keep your own outputs - they feed your final slides.

In [ ]:
import sqlite3
import pandas as pd

df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv")
df.columns = [c.strip() for c in df.columns]
print(df.columns.tolist())
df.head()

Column names differ slightly between lab versions. Check the printout above, then edit the rename dictionary if needed.

In [ ]:
rename = {'Time (UTC)': 'Time'}
df = df.rename(columns=rename)
df['Date'] = pd.to_datetime(df['Date'], format='mixed', dayfirst=True).dt.strftime('%Y-%m-%d')
con = sqlite3.connect('my_spacex.db')
df.to_sql('SPACEXTBL', con, if_exists='replace', index=False)

def q(sql):
    return pd.read_sql_query(sql, con)

### Task 1 - Unique launch sites

In [ ]:
q('SELECT DISTINCT Launch_Site FROM SPACEXTBL')

### Task 2 - Five records where the launch site starts with 'CCA'

In [ ]:
q("SELECT * FROM SPACEXTBL WHERE Launch_Site LIKE 'CCA%' LIMIT 5")

### Task 3 - Total payload mass carried for NASA (CRS)

In [ ]:
q("SELECT SUM(PAYLOAD_MASS__KG_) AS total_payload_kg FROM SPACEXTBL WHERE Customer LIKE 'NASA (CRS)'")

### Task 4 - Average payload mass for booster version F9 v1.1

In [ ]:
q("SELECT AVG(PAYLOAD_MASS__KG_) AS avg_payload_kg FROM SPACEXTBL WHERE Booster_Version LIKE 'F9 v1.1%'")

### Task 5 - Date of the first successful landing on a ground pad

In [ ]:
q("SELECT MIN(Date) AS first_ground_pad_success FROM SPACEXTBL WHERE Landing_Outcome LIKE 'Success (ground pad)'")

### Task 6 - Boosters with a successful drone-ship landing and payload between 4000 and 6000 kg

In [ ]:
q("SELECT DISTINCT Booster_Version FROM SPACEXTBL WHERE Landing_Outcome LIKE 'Success (drone ship)' AND PAYLOAD_MASS__KG_ BETWEEN 4000 AND 6000")

### Task 7 - Total number of each mission outcome

In [ ]:
q('SELECT Mission_Outcome, COUNT(*) AS n FROM SPACEXTBL GROUP BY Mission_Outcome')

### Task 8 - Boosters that carried the maximum payload

In [ ]:
q('SELECT DISTINCT Booster_Version FROM SPACEXTBL WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTBL)')

### Task 9 - 2015 drone-ship failures: month, landing outcome, booster, launch site

In [ ]:
q("""SELECT substr(Date, 6, 2) AS month, Landing_Outcome, Booster_Version, Launch_Site
     FROM SPACEXTBL
     WHERE Landing_Outcome LIKE 'Failure (drone ship)' AND substr(Date, 1, 4) = '2015'""")

### Task 10 - Rank landing outcomes between 2010-06-04 and 2017-03-20, descending

In [ ]:
q("""SELECT Landing_Outcome, COUNT(*) AS n
     FROM SPACEXTBL
     WHERE Date BETWEEN '2010-06-04' AND '2017-03-20'
     GROUP BY Landing_Outcome
     ORDER BY n DESC""")

## Write your own interpretation here
One or two sentences per task: what does each result tell you about SpaceX launches? Note anything in the text columns that needed cleaning.